<a href="https://colab.research.google.com/github/jorobledo/curso_aprendizaje_automatico/blob/master/practico/clase_4/Perceptron_multiple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Perceptrón Múltiple

Queremos predecir el precio de casas utilizando un perceptrón simple.

[Imagen de perceptrón múltiple](https://miro.medium.com/max/1138/1*MF1q2Q3fbpYlXX8fZUiwpA.png)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor


Metodologia para entrenar algoritmo:

In [ ]:
!pip install numpy

In [ ]:
# Base de datos para google collab:
!wget https://raw.githubusercontent.com/JoaquinAmatRodrigo/Estadistica-machine-learning-python/master/data/SaratogaHouses.csv

In [ ]:
# Base de datos para jupyter notebook
url = ("https://raw.githubusercontent.com/JoaquinAmatRodrigo/"
       "Estadistica-machine-learning-python/master/data/SaratogaHouses.csv")
datos = pd.read_csv(url, sep=",")

datos.columns = ["precio", "metros_totales", "antiguedad", "precio_terreno",
                 "metros_habitables", "universitarios", "dormitorios", 
                 "chimenea", "banyos", "habitaciones", "calefaccion",
                 "consumo_calefacion", "desague", "vistas_lago",
                 "nueva_construccion", "aire_acondicionado"]

In [ ]:
datos

Precios de casas de acuerdo a diversas caracterizticas. 

## Análisis exploratorio

In [ ]:
datos.info()

todas las columnas tienen el tipo de dato correcto

In [ ]:
datos.isna().sum().sort_values()

Cuantos datos faltantes tiene la base de datos. Luego de ver esto hay que tomar alguna desicion, imputar u otra. 

No hay valores ausentes en la base de datos

Queremos predecir el precio de las casas. Con lo cual vamos a ver cómo es la distribución de la variable respuesta: 

In [ ]:
plt.hist(datos.precio, bins=20)
plt.xlabel('Precio')
plt.ylabel('Cantidad')
plt.show()

Distribucion de la variable regresora. 

## División del conjunto de datos en entrenamiento y en prueba.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    datos.drop('precio', axis=1), #Tire variable precio 
    datos['precio'], # la variable dependiente es datos precio
    train_size = 0.8,
    random_state = 142,
    shuffle = True
)


In [ ]:
X_train.describe()

In [ ]:
X_test.describe()

Vemos que entrenamiento y testeo sean maso menos parecidos. 

## Preprocesamiento 
Los modelos de redes neuronales en general necesitan de preprocesamiento. Los dos más comunes son la binarización de las variables categóricas y la estandarización de las variables continuas. 

In [ ]:
numeric_cols = X_train.select_dtypes(include=['float64', 'int']).columns.to_list()
numeric_cols

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.to_list()
cat_cols

Separa en variables continuas y categoricas. 

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# Transformaciones para las variables numéricas
numeric_transformer = Pipeline(
                        steps=[('scaler', StandardScaler())]
                      )

# Transformaciones para las variables categóricas
categorical_transformer = Pipeline(
                            steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))]
                          )

preprocessor = ColumnTransformer(
                    transformers=[
                        ('numeric', numeric_transformer, numeric_cols),
                        ('cat', categorical_transformer, cat_cols)
                    ],
                    remainder='passthrough'
                )

Hace la transformacion de las variables planteadas anteriormente. 
Lo hace a traves de un pipeline. 
Con el pipeline defino la secuencia de acciones que voy a aplicar a los datos.

In [ ]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

In [ ]:
X_train_prep

transformamos la salida en dataframe y añadimos el nombre de las columnas

Volvemos a convertir en un data frame y le pongo nomrbes de vuelta.

In [ ]:
encoded_cat = preprocessor.named_transformers_['cat']['onehot'].get_feature_names(cat_cols)
labels = np.concatenate([numeric_cols, encoded_cat])
datos_train_prep = pd.DataFrame(X_train_prep, columns=labels)
datos_train_prep.info()

In [ ]:
datos_train_prep

In [ ]:
modelo = MLPRegressor(activation='relu')

In [ ]:
param_distributions = {
    'hidden_layer_sizes': [(10), (20), (10, 10)],
    'alpha': np.logspace(-3, 3, 10),
    'learning_rate_init': [0.001, 0.01],
}

Hacemos cross validation de muchos parametros.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import multiprocessing

grid = RandomizedSearchCV(
        estimator  = modelo,
        param_distributions = param_distributions,
        n_iter     = 50,
        scoring    = 'neg_mean_squared_error', # Este es el criterio que le ponemos
        n_jobs     = multiprocessing.cpu_count() - 1, # Que use todos los cpu -1, para interaccion grafica.
        cv         = 2, 
        verbose    = 0,
        random_state = 123,
        return_train_score = True
       )

In [ ]:
grid.fit(X = datos_train_prep, y = y_train)

In [ ]:
resultados = pd.DataFrame(grid.cv_results_)

In [ ]:
resultados

In [ ]:
modelo_final = grid.best_estimator_

Ajusta el modelo con mejor combinacion de parametros

In [ ]:
modelo_final

In [ ]:
datos_test_prep = pd.DataFrame(X_test_prep, columns=labels)
predicciones = modelo_final.predict(X = datos_test_prep)

In [ ]:
from sklearn.metrics import mean_squared_error
rmse = mean_squared_error(
        y_true = y_test,
        y_pred = predicciones,
        squared = False
       )
rmse

In [ ]:
predicciones[0]

In [ ]:
y_test.iloc[1]

In [ ]:
y_test.shape

In [ ]:
i=20
y_test.iloc[i]-predicciones[i]

Cuanto le erra para cada prediccion, en un caso particular.

In [ ]:
modelo_final.get_params()

Permite obtener parametros y bueno si estamos de acuerdo lo guardamos y ya tenemos archivo con los pesos entrenados y puedo correrlo en cualquier lado para predecir. 